# 배치 이미지 생성 — scenes.csv 한 줄이 그림 한 장

표에 한 줄을 적으면 그림 한 장이 나온다. 여섯 칸(`subject` `action` `camera` `lighting`
`environment` `style`)을 영어 문장 하나로 이어 프롬프트를 만들고, `out/{id}.png` 와
`out/{id}.json` 을 남긴다.

## 먼저 GPU 를 켠다

위 메뉴에서 **런타임 → 런타임 유형 변경 → 하드웨어 가속기: T4 GPU** 를 고르고 저장한다.

`float16` 은 GPU 에서 쓰는 형식이라 CPU 로는 돌지 않는다. 안 켜고 돌리면 아래 첫 칸이
알려 준다.

## 사람이 정한 것 / 맡긴 것

| 정한 것 | 왜 |
|---|---|
| **입력 형식** — 일곱 칸 | 칸 이름이 바뀌면 프롬프트 뜻이 달라진다 |
| **모델과 값** — sdxl-turbo · 스텝 4 · 가이던스 0.0 · float16 | 이 모델은 적은 스텝용이라 값을 올리면 오히려 나빠진다 |
| **재현 기록** — `out/{id}.json` | 이게 없으면 잘 나온 그림을 두 번 만들 수 없다 |
| **실패 처리** — 건너뛰고 끝에 목록 | 한 줄 때문에 올려 둔 모델과 지금까지 만든 것을 버리지 않는다 |

반복문을 어떻게 짜고 파일을 어떻게 쓸지는 정하지 않았다 — 바뀌어도 결과가 같다.

## 0. GPU 가 켜져 있는지 본다

In [ ]:
import torch

print("torch", torch.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU 를 못 찾았습니다.")
    print("런타임 → 런타임 유형 변경 → 하드웨어 가속기: T4 GPU 로 바꾸고 다시 실행하세요.")

## 1. 필요한 것 설치

`torch` 는 코랩에 이미 깔려 있다. 그림 모델을 다루는 `diffusers` 만 더 받으면 된다.
(1분쯤 걸린다.)

In [ ]:
!pip install -q diffusers transformers accelerate safetensors

## 2. 입력 — scenes.csv

아래 칸이 예시 표를 만든다. **내 표를 쓰려면** 왼쪽 파일 창에 `scenes.csv` 를 올리고
이 칸을 건너뛰면 된다.

| 열 | 뜻 | 예 |
|---|---|---|
| `id` | 파일 이름이자 시드의 재료. **겹치면 안 된다** | `fox-01` |
| `subject` | 무엇이 | `a red fox` |
| `action` | 무엇을 하는가 | `leaping over a fallen log` |
| `camera` | 어떻게 찍었는가 | `low-angle close-up shot` |
| `lighting` | 빛 | `golden hour backlight` |
| `environment` | 어디인가 | `misty pine forest` |
| `style` | 그림체 | `cinematic photorealistic` |

In [ ]:
from pathlib import Path

SCENES = """id,subject,action,camera,lighting,environment,style
fox-01,a red fox,leaping over a fallen log,low-angle close-up shot,golden hour backlight,misty pine forest,cinematic photorealistic
cat-desk-01,a tabby cat,sitting at a desk looking at a laptop,eye-level medium shot,soft morning window light,tidy home office,flat vector illustration
harbor-01,a small fishing boat,returning to the pier,wide establishing shot,blue hour twilight,quiet northern harbor,muted film photography
market-01,an elderly baker,arranging bread on a tray,over-the-shoulder shot,warm tungsten light,narrow old town bakery,documentary photography
robot-01,a small repair robot,tightening a bolt on a pipe,macro close-up,cool blue rim light,cluttered engine room,detailed concept art
"""

Path("scenes.csv").write_text(SCENES, encoding="utf-8")
print(Path("scenes.csv").read_text(encoding="utf-8"))

## 3. 정한 값과, 줄을 프롬프트로 바꾸는 규칙

여기가 **맡기면 안 되는 부분**이다. 바꾸고 싶으면 이 칸만 고친다.

In [ ]:
import csv
import hashlib
import json

# ── 입력 형식 ────────────────────────────────────────────
ID_COLUMN = "id"
PROMPT_COLUMNS = ["subject", "action", "camera", "lighting", "environment", "style"]
COLUMNS = [ID_COLUMN] + PROMPT_COLUMNS

# ── 모델과 값 ────────────────────────────────────────────
MODEL_ID = "stabilityai/sdxl-turbo"
STEPS = 4
GUIDANCE = 0.0
DTYPE = "float16"


def build_prompt(row):
    """여섯 칸을 영어 문장 하나로 잇는다.

    "무엇이 + 무엇을 한다" 를 앞에 놓고, 찍는 법·빛·장소·그림체를 쉼표로 덧붙인다.
    빈 칸은 건너뛴다 — 쉼표만 둘씩 이어지면 모델이 헷갈린다.
    """
    subject = row.get("subject", "").strip()
    action = row.get("action", "").strip()

    head = " ".join(p for p in (subject, action) if p)
    tail = [row.get(c, "").strip() for c in ("camera", "lighting", "environment", "style")]

    parts = [p for p in [head] + tail if p]
    if not parts:
        raise ValueError("여섯 칸이 모두 비어 있어 프롬프트를 만들 수 없습니다")
    return ", ".join(parts) + "."


def seed_from_id(scene_id):
    """id 를 언제 돌려도 같은 숫자 하나로 바꾼다.

    파이썬 기본 hash() 를 쓰면 안 된다. 실행할 때마다 값이 달라져서(해시 랜덤화)
    같은 id 인데 어제와 오늘 다른 그림이 나온다 — 재현이 깨진다.
    sha256 은 어느 컴퓨터에서 언제 돌려도 같은 값이라 여기에 쓴다.
    """
    digest = hashlib.sha256(scene_id.encode("utf-8")).digest()
    return int.from_bytes(digest[:4], "big")      # 0 ~ 4,294,967,295


def load_scenes(path="scenes.csv"):
    """표를 읽고, 그림을 만들기 전에 어긋난 곳을 먼저 잡는다."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"입력 파일이 없습니다: {path}")

    # utf-8-sig: 엑셀이 저장한 csv 는 맨 앞에 보이지 않는 표식(BOM)이 붙는다.
    # 그냥 utf-8 로 읽으면 첫 열 이름이 BOM+id 가 되어 id 를 못 찾는다.
    with path.open(encoding="utf-8-sig", newline="") as f:
        rows = list(csv.DictReader(f))

    if not rows:
        raise ValueError(f"읽을 줄이 없습니다: {path}")

    missing = [c for c in COLUMNS if c not in rows[0]]
    if missing:
        raise ValueError(f"열이 모자랍니다: {', '.join(missing)}")

    ids = [r[ID_COLUMN].strip() for r in rows]
    if not all(ids):
        raise ValueError("id 가 빈 줄이 있습니다")
    dup = {i for i in ids if ids.count(i) > 1}
    if dup:
        # 같은 id 가 둘이면 뒤엣것이 앞엣것을 덮어쓴다. 조용히 사라지느니 먼저 막는다.
        raise ValueError(f"id 가 겹칩니다: {', '.join(sorted(dup))}")

    return rows


print("정해 둔 값:", MODEL_ID, "· 스텝", STEPS, "· 가이던스", GUIDANCE, "·", DTYPE)

## 4. 그림을 만들기 전에 말부터 확인

모델을 올리는 데 몇 분이 걸린다. **그 전에** 프롬프트가 제대로 조립되는지, 시드가 무엇인지
눈으로 본다. 여기서 이상하면 위 표를 고치고 다시 실행한다.

In [ ]:
rows = load_scenes("scenes.csv")

for row in rows:
    print(f"{row['id']}  seed={seed_from_id(row['id'])}")
    print("   ", build_prompt(row))

## 5. 모델을 GPU 에 **한 번만** 올린다

이 한 번이 몇 분 걸린다(처음에는 모델을 내려받는다). 줄마다 올렸다 내렸다 하면
그 시간이 줄 수만큼 곱해진다. 그래서 따로 떼어 둔 칸이다.

In [ ]:
from diffusers import AutoPipelineForText2Image

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU(CUDA)를 찾지 못했습니다. float16 은 GPU 형식이라 CPU 로는 돌지 않습니다. "
        "런타임 → 런타임 유형 변경 → T4 GPU 로 바꾸고 처음부터 다시 실행하세요."
    )

try:
    pipe = AutoPipelineForText2Image.from_pretrained(
        MODEL_ID, torch_dtype=getattr(torch, DTYPE), variant="fp16"
    )
except Exception:
    # fp16 전용 파일이 없는 저장소도 있다. 그때는 보통 파일을 받아 float16 으로 올린다.
    pipe = AutoPipelineForText2Image.from_pretrained(
        MODEL_ID, torch_dtype=getattr(torch, DTYPE)
    )

pipe = pipe.to("cuda")
print("올렸습니다:", MODEL_ID)

## 6. 줄을 순회하며 만든다

한 줄이 넘어져도 멈추지 않는다. 건너뛰고 끝까지 간 뒤, 마지막에 실패 목록을 한꺼번에 알린다.

In [ ]:
OUT = Path("out")
OUT.mkdir(exist_ok=True)

failed = []

for n, row in enumerate(rows, 1):
    scene_id = row[ID_COLUMN].strip()
    try:
        prompt = build_prompt(row)
        seed = seed_from_id(scene_id)

        # 시드를 넣은 난수기를 줄마다 새로 만든다.
        # 돌려쓰면 앞 줄이 난수기를 얼마나 썼는지에 따라 결과가 달라진다.
        generator = torch.Generator(device="cuda").manual_seed(seed)

        image = pipe(
            prompt=prompt,
            num_inference_steps=STEPS,
            guidance_scale=GUIDANCE,
            generator=generator,
        ).images[0]

        image.save(OUT / f"{scene_id}.png")

        # ── 재현 기록 ────────────────────────────────────
        # 그림만 남기면 나중에 "이거 어떻게 나왔지?" 를 알 수 없다.
        record = {
            "id": scene_id,
            "prompt": prompt,
            "seed": seed,
            "model": MODEL_ID,
            "steps": STEPS,
            "guidance_scale": GUIDANCE,
            "dtype": DTYPE,
        }
        (OUT / f"{scene_id}.json").write_text(
            json.dumps(record, ensure_ascii=False, indent=2), encoding="utf-8"
        )

        print(f"[{n}/{len(rows)}] {scene_id}  seed={seed}")

    except Exception as e:
        failed.append((scene_id, f"{type(e).__name__}: {e}"))
        print(f"[{n}/{len(rows)}] {scene_id}  실패 — 건너뜁니다")

print(f"끝났습니다. 성공 {len(rows) - len(failed)} · 실패 {len(failed)} → {OUT.resolve()}")

if failed:
    print("실패한 줄:")
    for scene_id, reason in failed:
        print(f"  {scene_id}: {reason}")

## 7. 결과 보기

In [ ]:
from IPython.display import display
from PIL import Image

for png in sorted(OUT.glob("*.png")):
    record = json.loads((OUT / f"{png.stem}.json").read_text(encoding="utf-8"))
    print(f"{record['id']}  seed={record['seed']}")
    print("   ", record["prompt"])
    display(Image.open(png).resize((320, 320)))

## 8. 내려받기

`out` 을 통째로 압축해 받는다. 압축 파일에 그림과 기록(json)이 함께 들어 있다.

In [ ]:
import shutil

shutil.make_archive("out", "zip", OUT)
print("out.zip", Path("out.zip").stat().st_size // 1024, "KB")

from google.colab import files

files.download("out.zip")

## 알아 둘 것

**같은 id 면 언제나 같은 그림이 나온다.** `id` 를 sha256 으로 숫자 하나로 바꾸기 때문이다.

```
"fox-01" → sha256 → 앞 4바이트 → 2115970659
```

파이썬 기본 `hash()` 를 썼다면 실행할 때마다 값이 달라져서(해시 랜덤화) 같은 표로도
어제와 오늘 다른 그림이 나온다. 그래서 쓰지 않았다.

**코랩은 한동안 안 쓰면 연결이 끊긴다.** 끊기면 `out/` 도 사라지므로, 만든 뒤에는 8번 칸으로
내려받아 둔다.

**모델 라이선스를 확인한다.** `sdxl-turbo` 는 비상업 연구용 라이선스다. 결과물을 어디에
쓸지에 따라 확인이 필요하다.